# D01 — Model Acquisition and Embedding Extraction

Downloads the three single-cell foundation model checkpoints and extracts
static gene embedding matrices used throughout the analysis.

**Expected directory layout (all paths relative to this notebooks/ directory):**
```
./
├── Geneformer/                    (auto-cloned from HuggingFace)
├── scGPT_human/                   (manual download from GitHub)
│   ├── best_model.pt
│   ├── args.json
│   └── vocab.json
├── scFoundation_model/            (manual download from GitHub)
│   ├── models.ckpt (~2 GB)
│   └── ...
├── scFoundation_repo/             (manual download from GitHub)
│   ├── OS_scRNA_gene_index.19264.tsv
│   └── ...
└── data/
    ├── gene_embeddings.npy        (output: Geneformer, 20,275 × 768)
    ├── scgpt_gene_embeddings.npy  (output: scGPT, 60,694 × 512)
    ├── sf_gene_embedding_geometry.csv  (output: scFoundation, 19,264 × 768)
    ├── gene_names.json
    └── scgpt_gene_names.json
```

**Outputs:**
- `data/gene_embeddings.npy` — Geneformer V2-104M (20,275 × 768)
- `data/scgpt_gene_embeddings.npy` — scGPT (60,694 × 512)
- `data/sf_gene_embedding_geometry.csv` — scFoundation (19,264 × 768, geometry pre-computed)
- `data/gene_names.json`, `data/scgpt_gene_names.json` — gene name lists

**Prerequisites:** PyTorch, transformers, geneformer, scgpt packages

## Section 1: Geneformer V2-104M

Clone from HuggingFace to obtain the model checkpoint. Geneformer is a BERT-based architecture with 12 layers and 768 hidden dimensions. The model vocabulary covers approximately 20,000 genes based on Ensembl gene identifiers.

In [1]:
from pathlib import Path
import subprocess
import shutil

GF_REPO = Path('Geneformer')

# Check git-lfs is installed (required for HuggingFace model weights)
if not GF_REPO.exists():
    if shutil.which('git-lfs') is None:
        result = subprocess.run(['git', 'lfs', 'version'], capture_output=True)
        if result.returncode != 0:
            raise RuntimeError(
                'git-lfs is not installed. Geneformer model weights are stored as '
                'Git LFS objects on HuggingFace and cannot be cloned without it.\n'
                'Install with: brew install git-lfs && git lfs install  (macOS)\n'
                'Or see: https://git-lfs.github.com/'
            )

if not GF_REPO.exists():
    print('Cloning Geneformer repository from HuggingFace...')
    print('  (This includes ~2 GB of model weights via Git LFS)')
    subprocess.run(
        ['git', 'clone', 'https://huggingface.co/ctheodoris/Geneformer', str(GF_REPO)],
        check=True
    )
    print(f'Cloned to {GF_REPO}')
else:
    print(f'Geneformer already present at {GF_REPO}')

# Verify LFS files were actually pulled (not just pointer files)
config_file = GF_REPO / 'Geneformer-V2-104M' / 'config.json'
if config_file.exists() and config_file.stat().st_size < 200:
    with open(config_file) as f:
        content = f.read()
    if 'version https://git-lfs.github.com' in content:
        print('WARNING: LFS pointer files detected — pulling actual model weights...')
        subprocess.run(['git', 'lfs', 'pull'], cwd=str(GF_REPO), check=True)
        print('LFS pull complete.')


Geneformer already present at Geneformer


In [2]:
import numpy as np
import json
import pickle

MODEL_DIR = GF_REPO / 'Geneformer-V2-104M'
TOKEN_DICT = GF_REPO / 'geneformer' / 'token_dictionary_gc104M.pkl'

Path('data').mkdir(parents=True, exist_ok=True)
EMB_OUT = Path('data/gene_embeddings.npy')
NAMES_OUT = Path('data/gene_names.json')

if EMB_OUT.exists() and NAMES_OUT.exists():
    gf_emb = np.load(EMB_OUT)
    gf_names = json.load(open(NAMES_OUT))
    print(f'Loaded cached Geneformer embeddings: {gf_emb.shape}')
else:
    from transformers import BertModel
    
    model = BertModel.from_pretrained(str(MODEL_DIR))
    emb_matrix = model.embeddings.word_embeddings.weight.detach().cpu().numpy()
    
    with open(TOKEN_DICT, 'rb') as f:
        token_dict = pickle.load(f)
    
    # token_dict maps ensembl_id -> token_id
    # Invert to get token_id -> ensembl_id
    id_to_gene = {v: k for k, v in token_dict.items()}
    
    # Extract only gene tokens (skip special tokens)
    gene_ids = sorted([tid for tid in id_to_gene.keys() if isinstance(tid, int) and tid < emb_matrix.shape[0]])
    gene_names = [id_to_gene[tid] for tid in gene_ids]
    gene_embeddings = emb_matrix[gene_ids]
    
    np.save(EMB_OUT, gene_embeddings)
    with open(NAMES_OUT, 'w') as f:
        json.dump(gene_names, f)
    
    print(f'Extracted Geneformer embeddings: {gene_embeddings.shape}')
    print(f'Saved to {EMB_OUT} and {NAMES_OUT}')


Loaded cached Geneformer embeddings: (20275, 768)


## Section 2: scGPT

Download scGPT_human from https://github.com/bowang-lab/scGPT → Releases. Extract to scGPT_human/ (should contain best_model.pt, args.json, vocab.json).

scGPT uses a GPT-style transformer architecture with 512 hidden dimensions and supports a vocabulary of approximately 60,000 genes. Model files must be manually downloaded from GitHub releases.

In [3]:
SCGPT_DIR = Path('scGPT_human')

EMB_OUT_SC = Path('data/scgpt_gene_embeddings.npy')
NAMES_OUT_SC = Path('data/scgpt_gene_names.json')

if EMB_OUT_SC.exists() and NAMES_OUT_SC.exists():
    sc_emb = np.load(EMB_OUT_SC)
    sc_names = json.load(open(NAMES_OUT_SC))
    print(f'Loaded cached scGPT embeddings: {sc_emb.shape}')
elif not SCGPT_DIR.exists():
    print('scGPT checkpoint not found — skipping extraction.')
    print(f'Expected directory: {SCGPT_DIR}/')
    print('Download from: https://github.com/bowang-lab/scGPT → Releases → scGPT_human.zip')
else:
    # Full extraction from scGPT checkpoint
    import torch
    from scgpt.model import TransformerModel
    from scgpt.tokenizer.gene_tokenizer import GeneVocab

    vocab = GeneVocab.from_file(str(SCGPT_DIR / 'vocab.json'))
    print(f'Vocab size: {len(vocab):,} gene symbols')

    special_tokens = ['<pad>', '<cls>', '<eoc>']
    for st in special_tokens:
        if st not in vocab:
            vocab.append_token(st)

    PAD_TOKEN = '<pad>'
    CLS_TOKEN = '<cls>'

    with open(SCGPT_DIR / 'args.json') as f:
        model_args = json.load(f)

    D_MODEL  = model_args.get('embsize',  512)
    N_HEADS  = model_args.get('nheads',   8)
    D_HID    = model_args.get('d_hid',    512)
    N_LAYERS = model_args.get('nlayers',  12)
    print(f'd_model={D_MODEL}  nheads={N_HEADS}  nlayers={N_LAYERS}')

    model = TransformerModel(
        ntoken=len(vocab), d_model=D_MODEL, nhead=N_HEADS,
        d_hid=D_HID, nlayers=N_LAYERS,
        nlayers_cls=model_args.get('n_cls_layer', 3), n_cls=1,
        vocab=vocab, dropout=0.0, pad_token=PAD_TOKEN,
        pad_value=model_args.get('pad_value', 0),
        do_mvc=False, do_dab=False, use_batch_labels=False,
        domain_spec_batchnorm=False, explicit_zero_prob=False,
        use_fast_transformer=model_args.get('fast_transformer', False),
        pre_norm=model_args.get('pre_norm', False),
    )

    state = torch.load(SCGPT_DIR / 'best_model.pt', map_location='cpu')
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters')

    # Extract embedding matrix
    with torch.no_grad():
        emb_matrix = model.encoder.embedding.weight.detach().cpu().float().numpy()

    # Build gene list (exclude special tokens)
    gene_names_sc = []
    gene_ids_sc = []
    for gene_sym, idx in vocab.get_stoi().items():
        if gene_sym.startswith('<'):
            continue
        gene_names_sc.append(gene_sym)
        gene_ids_sc.append(idx)

    gene_ids_arr = np.array(gene_ids_sc, dtype=int)
    sc_emb = emb_matrix[gene_ids_arr]
    sc_names = gene_names_sc

    print(f'Gene embeddings: {sc_emb.shape} ({len(sc_names)} genes)')

    # Save
    np.save(EMB_OUT_SC, sc_emb)
    with open(NAMES_OUT_SC, 'w') as f:
        json.dump(sc_names, f)
    print(f'Saved: {EMB_OUT_SC}, {NAMES_OUT_SC}')

Loaded cached scGPT embeddings: (60694, 512)


## Section 3: scFoundation (xTrimoGene)

Download the xTrimoGene checkpoint from https://github.com/biomap-research/scFoundation. Place models.ckpt (~2 GB) in scFoundation_model/. Also download OS_scRNA_gene_index.19264.tsv from the same repository.

scFoundation leverages positional embeddings as gene representations, yielding 768-dimensional embeddings for 19,264 genes. The model checkpoint is approximately 2 GB in size.

In [4]:
SF_CKPT = Path('scFoundation_model/models.ckpt')
SF_GENES = Path('scFoundation_repo/OS_scRNA_gene_index.19264.tsv')
SF_OUT = Path('data/sf_gene_embedding_geometry.csv')

# Auto-download the gene index file if missing (small TSV, ~300 KB)
if not SF_GENES.exists() and SF_CKPT.exists():
    import urllib.request
    SF_GENES.parent.mkdir(parents=True, exist_ok=True)
    gene_index_url = (
        'https://raw.githubusercontent.com/biomap-research/scFoundation/'
        'main/model/OS_scRNA_gene_index.19264.tsv'
    )
    print(f'Downloading scFoundation gene index from GitHub...')
    try:
        urllib.request.urlretrieve(gene_index_url, str(SF_GENES))
        print(f'Saved: {SF_GENES}')
    except Exception as e:
        print(f'Auto-download failed ({e}). Please manually download:')
        print(f'  {gene_index_url}')
        print(f'  → Place in {SF_GENES}')

if SF_OUT.exists():
    import pandas as pd
    sf_df = pd.read_csv(SF_OUT)
    print(f'Loaded cached scFoundation geometry: {sf_df.shape}')
elif not SF_CKPT.exists():
    print('scFoundation checkpoint not found — skipping extraction.')
    print(f'Expected checkpoint: {SF_CKPT}')
    print('Download from: https://github.com/biomap-research/scFoundation')
elif not SF_GENES.exists():
    print('scFoundation gene list not found — skipping extraction.')
    print(f'Expected file: {SF_GENES}')
    print('Download from: https://github.com/biomap-research/scFoundation')
else:
    import torch
    import pandas as pd
    import collections
    from sklearn.neighbors import NearestNeighbors
    from sklearn.decomposition import PCA
    from scipy.stats import zscore

    # Load checkpoint and extract gene embeddings
    print('Loading scFoundation checkpoint...')
    ckpt = torch.load(str(SF_CKPT), map_location='cpu')
    state_dict = ckpt['gene']['state_dict']

    # Strip 'model.' prefix from MMF keys
    sd = collections.OrderedDict()
    for k, v in state_dict.items():
        new_key = k.split('model.')[1] if 'model.' in k else k
        sd[new_key] = v

    # pos_emb.weight: (19267, 768) — first 19264 are gene embeddings
    pos_emb_full = sd['pos_emb.weight'].float().numpy()
    emb_matrix = pos_emb_full[:19264]
    print(f'Extracted embeddings: {emb_matrix.shape}')

    # Load gene names
    gene_list_df = pd.read_csv(str(SF_GENES), sep='\t', header=0)
    gene_list_df = gene_list_df.sort_values('index').reset_index(drop=True)
    gene_symbols = gene_list_df['gene_name'].tolist()
    assert len(gene_symbols) == 19264

    # Compute geometry metrics (same as P01)
    norms = np.linalg.norm(emb_matrix, axis=1)
    centroid = emb_matrix.mean(axis=0)
    dist_to_centroid = np.linalg.norm(emb_matrix - centroid, axis=1)
    centroid_norm = np.linalg.norm(centroid)
    cos_to_centroid = (emb_matrix @ centroid) / (norms * centroid_norm + 1e-12)

    # Isolation score: mean cosine distance to k=10 nearest neighbours
    print('Computing nearest neighbours (k=10)...')
    nn = NearestNeighbors(n_neighbors=11, metric='cosine', n_jobs=-1)
    nn.fit(emb_matrix)
    distances, _ = nn.kneighbors(emb_matrix)
    isolation = distances[:, 1:].mean(axis=1)

    # Z-scores and anomaly
    Z_THRESH = 3.0
    norm_z = zscore(norms)
    dist_z = zscore(dist_to_centroid)
    cos_z = zscore(cos_to_centroid)
    iso_z = zscore(isolation)
    anomaly = np.maximum.reduce([np.abs(norm_z), np.abs(dist_z),
                                  np.abs(cos_z), np.abs(iso_z)])

    pca = PCA(n_components=2, random_state=42)
    pca_coords = pca.fit_transform(emb_matrix - centroid)

    sf_df = pd.DataFrame({
        'gene': gene_symbols,
        'token_id': np.arange(len(gene_symbols)),
        'norm': norms,
        'dist_from_centroid': dist_to_centroid,
        'cos_to_centroid': cos_to_centroid,
        'norm_zscore': norm_z,
        'dist_zscore': dist_z,
        'cos_zscore': cos_z,
        'is_outlier': anomaly > Z_THRESH,
        'pca_1': pca_coords[:, 0],
        'pca_2': pca_coords[:, 1],
        'isolation_score': isolation,
        'isolation_zscore': iso_z,
        'anomaly_score': anomaly,
    })

    sf_df.to_csv(SF_OUT, index=False)
    print(f'Saved: {SF_OUT} ({len(sf_df)} genes, {sf_df["is_outlier"].sum()} outliers)')


Loaded cached scFoundation geometry: (19264, 14)


## Section 4: Verification

Verify that all expected output files have been created and are ready for downstream analysis.

In [5]:
from pathlib import Path

expected = {
    'Geneformer embeddings': Path('data/gene_embeddings.npy'),
    'Geneformer gene names': Path('data/gene_names.json'),
    'scGPT embeddings': Path('data/scgpt_gene_embeddings.npy'),
    'scGPT gene names': Path('data/scgpt_gene_names.json'),
    'scFoundation geometry': Path('data/sf_gene_embedding_geometry.csv'),
}

print('D01 Output Verification')
print('=' * 60)
all_ok = True
for name, path in expected.items():
    exists = path.exists()
    size = f'{path.stat().st_size / 1e6:.1f} MB' if exists else 'MISSING'
    status = 'OK' if exists else 'XX'
    print(f'  [{status}] {name:30s}  {size}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll outputs present. Ready for P01_embedding_geometry.')
else:
    print('\nSome outputs missing — see instructions above for extraction.')

# ── Generate data manifest for reproducibility ──────────────────────
import hashlib, json as _json
from datetime import datetime

def sha256_of(path, chunk=65536):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            data = f.read(chunk)
            if not data:
                break
            h.update(data)
    return h.hexdigest()

manifest = {
    'created': datetime.now().isoformat(),
    'models': {
        'geneformer': {
            'source': 'HuggingFace ctheodoris/Geneformer',
            'version': 'V2-104M (GC-104M)',
            'vocab_size': 20275,
            'embed_dims': 768,
        },
    },
    'files': {}
}

for name, path in expected.items():
    if path.exists():
        manifest['files'][name] = {
            'path': str(path),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_of(path) if path.stat().st_size < 500_000_000 else 'skipped (>500MB)',
        }

# Add model info based on what files are present
if Path('data/scgpt_gene_embeddings.npy').exists():
    manifest['models']['scgpt'] = {
        'source': 'scGPT package (best_model.pt)',
        'vocab_size': 60694,
        'embed_dims': 512,
    }

if Path('data/sf_gene_embedding_geometry.csv').exists():
    manifest['models']['scfoundation'] = {
        'source': 'HuggingFace genbio-ai/scFoundation',
        'vocab_size': 19264,
        'embed_dims': 768,
    }

manifest_path = Path('data/manifest.json')
with open(manifest_path, 'w') as f:
    _json.dump(manifest, f, indent=2)
print(f'\nSaved: {manifest_path}')

D01 Output Verification
  [OK] Geneformer embeddings           62.3 MB
  [OK] Geneformer gene names           0.4 MB
  [OK] scGPT embeddings                124.3 MB
  [OK] scGPT gene names                0.8 MB
  [OK] scFoundation geometry           2.6 MB

All outputs present. Ready for P01_embedding_geometry.

Saved: data/manifest.json
